# Model-family walkthrough

This notebook presents the 14 pre-registered model trials used by the benchmark. Each model has one executable cell, and each cell is preceded by a short explanation.

The families are grouped in a fixed order. Within each family, variants run from smaller to larger capacity. Every model uses the shared fixed raw split, selects its best epoch on validation macro-F1, saves a resumable checkpoint, and defers the held-out test.

Set COMPCLASS_PROFILE=smoke for a fast pipeline check. Set COMPCLASS_WALKTHROUGH_ARTIFACT_PREFIX to choose a new checkpoint prefix; the default family_walkthrough does not overwrite earlier benchmark artifacts.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK_DIR = Path(os.getenv("COMPCLASS_WORK_DIR", Path.cwd())).expanduser().resolve()
NOTEBOOK_DIR = WORK_DIR if (WORK_DIR / "12_model_family_benchmark.py").exists() else WORK_DIR / "notebooks"
RUNNER_PATH = NOTEBOOK_DIR / "12_model_family_benchmark.py"
if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Could not find {RUNNER_PATH}")

PROFILE_DEFAULTS = {"smoke": (1, 1), "standard": (10, 3), "paper": (25, 5)}
PROFILE = os.getenv("COMPCLASS_PROFILE", "paper")
if PROFILE not in PROFILE_DEFAULTS:
    raise ValueError(f"Unknown COMPCLASS_PROFILE={PROFILE!r}")
EPOCHS = int(os.getenv("COMPCLASS_WALKTHROUGH_EPOCHS", PROFILE_DEFAULTS[PROFILE][0]))
PATIENCE = int(os.getenv("COMPCLASS_WALKTHROUGH_PATIENCE", PROFILE_DEFAULTS[PROFILE][1]))
RECIPE = os.getenv("COMPCLASS_WALKTHROUGH_RECIPE", "robust-v2-tta")
ARTIFACT_PREFIX = os.getenv("COMPCLASS_WALKTHROUGH_ARTIFACT_PREFIX", "family_walkthrough")
METRICS_DIR = NOTEBOOK_DIR / "outputs" / "metrics"

MODEL_GROUPS = {
    "ResNet": ["resnet34", "resnet101", "resnet152"],
    "DenseNet": ["densenet121"],
    "EfficientNet": ["efficientnet_b2", "efficientnet_v2_s", "efficientnet_v2_m"],
    "MobileNet": ["mobilenet_v3_large"],
    "ConvNeXt": ["convnext_tiny", "convnext_small", "convnext_base"],
    "Swin": ["swin_t", "swin_v2_t", "swin_b"],
}
MODEL_ORDER = [name for names in MODEL_GROUPS.values() for name in names]
MODEL_FAMILY = {name: family for family, names in MODEL_GROUPS.items() for name in names}
MODEL_RECORDS = {}
REPORT_PREFIX = f"model_walkthrough_{RECIPE.replace('-', '_')}"

def _report_path(model):
    return METRICS_DIR / f"{REPORT_PREFIX}_{model}.json"

def _row(record):
    training = record.get("training", {})
    validation = record.get("val", {})
    return {
        "family": MODEL_FAMILY[record["name"]],
        "model": record["name"],
        "status": "complete",
        "parameters": record.get("params", training.get("params", float("nan"))),
        "best_epoch": training.get("best_epoch", float("nan")),
        "val_accuracy": validation.get("accuracy", float("nan")),
        "val_macro_f1": validation.get("macro_f1", float("nan")),
        "val_weighted_f1": validation.get("weighted_f1", float("nan")),
        "train_accuracy": training.get("best_train_accuracy", float("nan")),
        "train_val_gap": training.get("train_val_accuracy_gap", float("nan")),
        "overfit_flag": record.get("overfit_flag", False),
        "wall_time_seconds": training.get("wall_time_seconds", float("nan")),
        "checkpoint": Path(record.get("checkpoint", "")).name,
    }

def _format_number(value, digits=4):
    return "—" if pd.isna(value) else f"{value:.{digits}f}"

def _display_record(record):
    row = pd.DataFrame([_row(record)])
    display(row.style.format({
        "parameters": lambda value: "—" if pd.isna(value) else f"{int(value):,}",
        "best_epoch": lambda value: "—" if pd.isna(value) else f"{int(value)}",
        "val_accuracy": _format_number,
        "val_macro_f1": _format_number,
        "val_weighted_f1": _format_number,
        "train_accuracy": _format_number,
        "train_val_gap": _format_number,
        "wall_time_seconds": lambda value: "—" if pd.isna(value) else f"{value:.1f}",
    }).background_gradient(subset=["val_accuracy", "val_macro_f1"], cmap="YlGn"))

def run_model(model):
    if model not in MODEL_ORDER:
        raise ValueError(f"Unknown walkthrough model: {model}")
    report_name = _report_path(model).name
    command = [
        sys.executable, str(RUNNER_PATH),
        "--models", model,
        "--profile", PROFILE,
        "--epochs", str(EPOCHS),
        "--patience", str(PATIENCE),
        "--recipe", RECIPE,
        "--artifact-prefix", ARTIFACT_PREFIX,
        "--report", report_name,
        "--defer-test",
    ]
    print(f"Running {model} | recipe={RECIPE} | profile={PROFILE} | report={report_name}")
    subprocess.run(
        command,
        cwd=NOTEBOOK_DIR,
        check=True,
        env={**os.environ, "COMPCLASS_WORK_DIR": str(NOTEBOOK_DIR)},
    )
    payload = json.loads(_report_path(model).read_text())
    trials = [trial for trial in payload.get("trials", []) if trial.get("name") == model]
    if len(trials) != 1:
        raise RuntimeError(f"Expected one completed trial for {model} in {_report_path(model)}")
    record = trials[0]
    MODEL_RECORDS[model] = record
    _display_record(record)
    return record

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Recipe: {RECIPE} | profile: {PROFILE} | epochs: {EPOCHS} | patience: {PATIENCE}")
print(f"Models: {len(MODEL_ORDER)} | held-out test: deferred")

## ResNet family

ResNet is a convolutional family built around residual skip connections, which let deep networks learn stable refinements.

**Pros:** mature transfer-learning behavior, stable optimization, and clear depth scaling.

**Cons:** deeper variants are expensive, can memorize noisy images, and mainly model local visual patterns.

### ResNet-34 — compact residual CNN

This smaller residual candidate establishes a lower-capacity baseline and tests the gain from moving beyond ResNet-18.

In [ ]:
run_model("resnet34")

### ResNet-101 — deep residual CNN

This deep residual candidate tests whether deeper hierarchical features resolve ambiguous component views better than compact ResNets.

In [ ]:
run_model("resnet101")

### ResNet-152 — largest residual CNN

This largest ResNet probes the upper end of residual capacity while validation and the gap guard protect against memorising noise.

In [ ]:
run_model("resnet152")

## DenseNet family

DenseNet connects each layer to later layers inside dense blocks, encouraging feature reuse instead of relearning features.

**Pros:** parameter-efficient feature reuse and strong fine-grained feature propagation.

**Cons:** dense concatenation increases memory traffic, training can be slower, and scaling is less straightforward.

### DenseNet-121 — dense feature reuse

This moderate-size candidate tests whether dense feature reuse helps more than residual depth on visually similar components.

In [ ]:
run_model("densenet121")

## EfficientNet family

EfficientNet scales depth, width, and input resolution together; EfficientNet-V2 modernizes that training and scaling recipe.

**Pros:** strong accuracy per parameter and useful capacity steps.

**Cons:** performance can be sensitive to recipe and resolution, while large variants become memory- and time-intensive.

### EfficientNet-B2 — efficient CNN baseline

This smaller EfficientNet tests whether compound scaling gives better accuracy per parameter than the other CNN families.

In [ ]:
run_model("efficientnet_b2")

### EfficientNet-V2-S — modern efficient CNN

This small-to-medium V2 model checks whether newer blocks and training design improve transfer on the noisy fixed split.

In [ ]:
run_model("efficientnet_v2_s")

### EfficientNet-V2-M — larger efficient CNN

This larger V2 candidate tests whether additional efficient capacity recovers difficult classes without crossing the overfit guard.

In [ ]:
run_model("efficientnet_v2_m")

## MobileNet family

MobileNet uses lightweight depthwise-separable convolutions and mobile-oriented blocks to reduce inference cost.

**Pros:** fast, compact, and practical for low-resource deployment.

**Cons:** lower capacity can hurt fine-grained or noisy classes and leaves less room for label ambiguity.

### MobileNet-V3-Large — lightweight CNN

This mobile-oriented model measures the accuracy cost of deployment-friendly design rather than maximum capacity.

In [ ]:
run_model("mobilenet_v3_large")

## ConvNeXt family

ConvNeXt is a modernized pure-convolutional family that adopts several design and training ideas associated with vision transformers.

**Pros:** strong pretrained transfer, competitive accuracy, simple convolutional inference, and a clean size ladder.

**Cons:** larger variants are memory-heavy and can amplify dataset noise without regularisation.

### ConvNeXt-Tiny — modern CNN baseline

This smallest ConvNeXt candidate tests whether modernised convolutional blocks transfer better than older CNN families.

In [ ]:
run_model("convnext_tiny")

### ConvNeXt-Small — medium modern CNN

This middle ConvNeXt size checks whether extra modern-CNN capacity improves macro-F1 without fitting dataset noise.

In [ ]:
run_model("convnext_small")

### ConvNeXt-Base — large modern CNN

This largest ConvNeXt tests whether more pretrained capacity improves difficult classes under regularised training.

In [ ]:
run_model("convnext_base")

## Swin family

Swin is a hierarchical vision transformer that uses local attention windows and shifts those windows between blocks.

**Pros:** multi-scale representations and broader spatial context than a conventional local convolution.

**Cons:** higher compute and memory cost, more training sensitivity, and no guarantee that attention helps a small noisy dataset.

### Swin-T — compact windowed transformer

This compact transformer baseline tests whether local-window attention and hierarchy beat convolutional inductive bias.

In [ ]:
run_model("swin_t")

### Swin-V2-T — updated compact transformer

This similarly compact V2 model separates the updated transformer design from simply buying a larger model.

In [ ]:
run_model("swin_v2_t")

### Swin-B — large windowed transformer

This largest transformer candidate tests the upper end of attention-based capacity with validation-only selection.

In [ ]:
run_model("swin_b")

In [ ]:
# Compile all per-model results into one ordered table and save it for later review.
rows = []
for model in MODEL_ORDER:
    if model in MODEL_RECORDS:
        rows.append(_row(MODEL_RECORDS[model]))
    else:
        rows.append({
            "family": MODEL_FAMILY[model],
            "model": model,
            "status": "not run",
            "parameters": float("nan"),
            "best_epoch": float("nan"),
            "val_accuracy": float("nan"),
            "val_macro_f1": float("nan"),
            "val_weighted_f1": float("nan"),
            "train_accuracy": float("nan"),
            "train_val_gap": float("nan"),
            "overfit_flag": None,
            "wall_time_seconds": float("nan"),
            "checkpoint": "",
        })

summary = pd.DataFrame(rows, columns=[
    "family", "model", "status", "parameters", "best_epoch",
    "val_accuracy", "val_macro_f1", "val_weighted_f1",
    "train_accuracy", "train_val_gap", "overfit_flag",
    "wall_time_seconds", "checkpoint",
])
display(summary.style.format({
    "parameters": lambda value: "—" if pd.isna(value) else f"{int(value):,}",
    "best_epoch": lambda value: "—" if pd.isna(value) else f"{int(value)}",
    "val_accuracy": _format_number,
    "val_macro_f1": _format_number,
    "val_weighted_f1": _format_number,
    "train_accuracy": _format_number,
    "train_val_gap": _format_number,
    "wall_time_seconds": lambda value: "—" if pd.isna(value) else f"{value:.1f}",
}).background_gradient(subset=["val_accuracy", "val_macro_f1"], cmap="YlGn"))

summary_dir = NOTEBOOK_DIR / "outputs" / "metrics"
summary_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(summary_dir / f"{REPORT_PREFIX}_summary.csv", index=False)
summary.to_json(summary_dir / f"{REPORT_PREFIX}_summary.json", orient="records", indent=2)

complete = summary[summary["status"] == "complete"]
missing = summary.loc[summary["status"] != "complete", "model"].tolist()
if missing:
    print(f"Run the remaining model cells before treating the summary as complete: {missing}")
if not complete.empty:
    family_summary = (
        complete.groupby("family", sort=False)
        .agg(
            models=("model", "count"),
            best_val_accuracy=("val_accuracy", "max"),
            best_val_macro_f1=("val_macro_f1", "max"),
            mean_train_val_gap=("train_val_gap", "mean"),
        )
        .reset_index()
    )
    display(family_summary.style.format({
        "best_val_accuracy": _format_number,
        "best_val_macro_f1": _format_number,
        "mean_train_val_gap": _format_number,
    }).background_gradient(subset=["best_val_accuracy", "best_val_macro_f1"], cmap="YlGn"))
print(f"Saved {len(complete)}/{len(summary)} model rows to {summary_dir / (REPORT_PREFIX + '_summary.csv')}")